# AML Multi-GNN — Training & Publishing
Trains the Multi-GNN edge classifier (Egressy et al., AAAI 2024 adaptations:
**reverse message passing + port numbering + ego IDs** on a BatchNorm-stabilized
GINE backbone), validates with **history-inclusive message passing** (paper
protocol), calibrates risk thresholds, publishes the model, and scores the
held-out test stream with history context.

GPU-first (fp32; AMP off by design). Early stopping: patience 4 on val PR-AUC.

**Prerequisites**
1. Data panel aliases: `gnn_training_code`, `raw_transactions_train`,
   `raw_transactions_val`, `raw_transactions_test_stream` (all in this project).
2. Models panel: model output alias matching the tier, e.g. `aml_gnn_hi_small`.
3. Packages: already configured — please don't add more.

In [ ]:
# Cell 2 — environment + runtime diagnostics
import importlib, shutil as _sh
required = {"torch": "pytorch", "torch_geometric": "pyg / torch-geometric",
            "pandas": "pandas", "numpy": "numpy", "sklearn": "scikit-learn",
            "palantir_models": "palantir_models"}
missing = [pkg for mod, pkg in required.items()
           if importlib.util.find_spec(mod) is None]
if missing:
    print("MISSING packages ", ", ".join(missing))
else:
    import torch
    print("environment OK | torch", torch.__version__,
          "| cuda:", torch.cuda.is_available(),
          "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

backend = next((m for m in ("pyg_lib", "torch_sparse")
                if importlib.util.find_spec(m) is not None), None)
print("sampling backend:", backend or "NONE — tell Claude!")
try:
    shm = _sh.disk_usage("/dev/shm")
    print(f"/dev/shm: {shm.total / 1e6:.0f} MB (small is fine — keep AML_NUM_WORKERS=0)")
except OSError:
    pass

In [ ]:
# Cell 3 — sync the training modules from the gnn_training_code dataset
import shutil
from pathlib import Path
from foundry.transforms import Dataset

MODULES = ["graph_builder.py", "model.py", "model_adapter.py",
           "training_core.py", "train_gnn_foundry.py", "score_large_tier.py"]
downloaded = Dataset.get("gnn_training_code").files().download()
for name, local_path in downloaded.items():
    shutil.copy(local_path, Path.cwd() / Path(name).name)
print("modules synced:", sorted(Path(m).name for m in MODULES if Path(m).exists()))

In [ ]:
# Cell 4 — configuration (presentable run; H100 profile)
import os
os.environ["AML_TIER"] = "HI-Small"      # HI-Small | HI-Medium | LI-Small | LI-Medium
os.environ["AML_EPOCHS"] = "40"          # upper bound; early stopping (AML_PATIENCE=6) decides
os.environ["AML_BATCH_SIZE"] = "8192"
os.environ["AML_FANOUT"] = "50,30"       # neighbors per hop; one entry per GNN layer.
                                         # Reference impl samples 100/hop; 15,10 was a T4 budget.
os.environ["AML_HIDDEN"] = "96"          # paper's GINe uses ~66; H100 has headroom
os.environ["AML_LAYERS"] = "2"           # must match len(AML_FANOUT)
os.environ["AML_LR"] = "1e-3"            # ReduceLROnPlateau halves it on 2 flat epochs
os.environ["AML_PATIENCE"] = "6"
os.environ.pop("AML_NUM_WORKERS", None)  # unset -> auto sampling threads (8)
# model output alias defaults to aml_gnn_<tier>; override with AML_MODEL_ALIAS
# AML_SKIP_PUBLISH=1 -> train + checkpoint only (publish later: publish_checkpoint.py)
print("tier:", os.environ["AML_TIER"],
      "| model alias: aml_gnn_" + os.environ["AML_TIER"].lower().replace("-", "_"),
      "| epochs<=", os.environ["AML_EPOCHS"], "| fanout", os.environ["AML_FANOUT"],
      "| hidden", os.environ["AML_HIDDEN"])

In [ ]:
%%keep_alive
# ^ Palantir cell magic: keeps this cell running up to 24h regardless of the
#   workspace idle timeout (default 30 min!). Output may stop displaying if the
#   tab is closed; progress is mirrored to ~/repo/train_<alias>.log.
# Cell 5 — train, calibrate thresholds, publish
# Trains the bidirectional GINE with ego IDs; validation uses the train+val
# graph for message passing (history-inclusive). Logs val ROC-AUC / PR-AUC /
# minority-F1 per epoch, keeps the best-PR-AUC weights, publishes, and
# returns the fitted adapter for cell 6.
import importlib, traceback
import graph_builder, model, model_adapter, training_core, train_gnn_foundry
for _m in (graph_builder, model, model_adapter, training_core, train_gnn_foundry):
    importlib.reload(_m)   # pick up cell-3 module updates + cell-4 env without a kernel restart

try:
    adapter = train_gnn_foundry.main()
except BaseException:
    # notebook outputs are stripped on commit; persist the traceback to a
    # file so it survives 'commit local changes' and can be debugged remotely
    with open("last_error.txt", "w") as fh:
        fh.write(traceback.format_exc())
    print("\nTraceback saved to last_error.txt — commit local changes to share it.")
    raise

In [ ]:
# Cell 6 — score the held-out 20% test stream WITH history context
# The demo artifact: every test transaction scored, message passing over the
# trailing history (val slice) so accounts keep their neighborhoods. Prints
# ROC-AUC, PR-AUC, and Precision@K — "of the top K alerts, how many are
# real" — the number your investigator workflow ultimately showcases.
import numpy as np
from foundry.transforms import Dataset
from sklearn.metrics import average_precision_score, roc_auc_score

test_df = Dataset.get("raw_transactions_test_stream").read_table(format="pandas")
history = Dataset.get("raw_transactions_val").read_table(format="pandas")
y_true = test_df["is_laundering"].astype(int).to_numpy()

scored = adapter.predict(test_df.drop(columns=["is_laundering"]), history_df=history)
y_score = scored["laundering_probability"].to_numpy()

order = np.argsort(-y_score)
print(f"test rows: {len(scored):,} | positives: {y_true.sum():,}")
print(f"ROC-AUC: {roc_auc_score(y_true, y_score):.4f} | "
      f"PR-AUC: {average_precision_score(y_true, y_score):.4f}")
for k in (100, 500, 1000, 5000):
    print(f"Precision@{k}: {y_true[order][:k].mean():.3f}")
print("risk bands:", scored["risk_band"].value_counts().to_dict())

scored["is_laundering_ground_truth"] = y_true
scored.to_parquet("scored_test_stream.parquet", index=False)
print("saved scored_test_stream.parquet — register a writable output dataset "
      "in the Data panel, then Dataset.get(<alias>).write_table(scored) to "
      "land it in Foundry for the ontology pipeline.")

# Large-tier scoring (HI-Large_Trans alias required) stays available via:
#   from score_large_tier import score_large
#   report = score_large(adapter, "HI-Large_Trans", max_parts=3)